In [1]:
# 06b-1. 기본 설정

from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

from sklearn.ensemble import RandomForestRegressor
from sklearn.dummy import DummyRegressor
from sklearn.model_selection import ParameterSampler
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)

PROJECT_ROOT = Path(
    r"C:\code\portfolio_optimization"
)

SUPERVISED_DATASET_PATH = (
    PROJECT_ROOT
    / "data"
    / "features"
    / "common"
    / "supervised_dataset.parquet"
)

print(
    SUPERVISED_DATASET_PATH.exists()
)

True


In [2]:
# 06b-2. supervised dataset 불러오기

supervised_dataset = (
    pq.read_table(
        SUPERVISED_DATASET_PATH
    )
    .to_pandas()
    .sort_values(
        [
            "signal_date",
            "ticker"
        ]
    )
    .reset_index(
        drop=True
    )
)

print(
    "shape:",
    supervised_dataset.shape
)

print(
    "signals:",
    supervised_dataset[
        "signal_date"
    ].nunique()
)

shape: (21779, 39)
signals: 436


In [3]:
# 06b-3. model feature 설정

ASSET_FEATURES = [
    "return_1d",
    "return_5d",
    "momentum_20d",
    "momentum_60d",
    "volatility_20d",
    "drawdown_20d",
    "trading_value_ma20",
    "trading_value_ratio_20d",
    "log_market_cap"
]

MARKET_FEATURES = [
    "market_return_1d",
    "market_return_5d",
    "market_return_20d",
    "market_volatility_20d",
    "market_drawdown",
    "volume_change_1d",
    "trading_value_change_1d",
    "market_trading_value_ratio_20d"
]

MACRO_FEATURES = [
    "base_rate",
    "usdkrw",
    "bond3y",
    "usdkrw_return_1d",
    "usdkrw_return_5d",
    "usdkrw_return_20d",
    "bond3y_change_1d",
    "bond3y_change_5d",
    "bond3y_change_20d",
    "base_rate_change",
    "rate_spread_3y"
]

MODEL_FEATURES = (
    ASSET_FEATURES
    + MARKET_FEATURES
    + MACRO_FEATURES
)

TARGET = "target_return"

print(
    len(MODEL_FEATURES)
)

28


In [4]:
# 06b-4. random forest 후보 고정

RF_PARAM_SPACE = {
    "max_depth": [
        4,
        8,
        12,
        None
    ],
    "min_samples_leaf": [
        0.001,
        0.0025,
        0.005,
        0.01
    ],
    "max_features": [
        "sqrt",
        0.5,
        1.0
    ]
}

In [5]:
# 06b-5. random forest 후보 sampling

RF_CANDIDATES = list(
    ParameterSampler(
        RF_PARAM_SPACE,
        n_iter=24,
        random_state=42
    )
)

print(
    "candidates:",
    len(RF_CANDIDATES)
)

print(
    RF_CANDIDATES[:5]
)

candidates: 24
[{'min_samples_leaf': 0.01, 'max_features': 'sqrt', 'max_depth': 12}, {'min_samples_leaf': 0.001, 'max_features': 0.5, 'max_depth': None}, {'min_samples_leaf': 0.005, 'max_features': 'sqrt', 'max_depth': 12}, {'min_samples_leaf': 0.01, 'max_features': 0.5, 'max_depth': None}, {'min_samples_leaf': 0.001, 'max_features': 'sqrt', 'max_depth': 12}]


24: 48개 전체 grid의 절반을 평가하면서 9-fold 계산량을 감당할 수 있게 줄이기 위한 실용적인 computational budget.
그리고 n_estimators는 이번 실험에서는 300으로 고정. 이것도 최적이라고 주장하는 숫자가 아니라, 이전 exploration에서 이미 안정적으로 실행됐고 tree 개수까지 search dimension에 넣으면 계산량이 크게 늘어나기 때문에 복잡도를 결정하는 depth/leaf/features에 우선 search budget을 쓰기 위한 선택임.

In [6]:
# 06b-6. cross-sectional ic 함수

def calculate_ic(
    data,
    prediction_column,
    target_column
):

    ic_values = []

    for _, group in data.groupby(
        "signal_date"
    ):

        if len(group) < 2:
            continue

        if (
            group[prediction_column].nunique() < 2
            or
            group[target_column].nunique() < 2
        ):
            continue

        ic = (
            group[
                prediction_column
            ]
            .rank()
            .corr(
                group[
                    target_column
                ]
                .rank()
            )
        )

        if pd.notna(ic):
            ic_values.append(ic)

    return np.array(
        ic_values
    )

In [7]:
# 06b-7. walk-forward 설정

TRAIN_YEARS = 3
VALIDATION_MONTHS = 6
TEST_MONTHS = 6
STEP_MONTHS = 6

In [8]:
# 06b-8. walk-forward fold 생성

signal_start = (
    supervised_dataset[
        "signal_date"
    ].min()
)

signal_end = (
    supervised_dataset[
        "signal_date"
    ].max()
)

folds = []

validation_start = (
    signal_start
    + pd.DateOffset(
        years=TRAIN_YEARS
    )
)

fold_id = 1


while True:

    test_start = (
        validation_start
        + pd.DateOffset(
            months=VALIDATION_MONTHS
        )
    )

    test_end = (
        test_start
        + pd.DateOffset(
            months=TEST_MONTHS
        )
    )

    if test_end > signal_end:
        break

    folds.append(
        {
            "fold": fold_id,
            "train_start": signal_start,
            "train_end": validation_start,
            "validation_start": validation_start,
            "validation_end": test_start,
            "test_start": test_start,
            "test_end": test_end
        }
    )

    validation_start = (
        validation_start
        + pd.DateOffset(
            months=STEP_MONTHS
        )
    )

    fold_id += 1


fold_table = pd.DataFrame(
    folds
)

print(
    fold_table.to_string(
        index=False
    )
)

 fold train_start  train_end validation_start validation_end test_start   test_end
    1  2018-05-04 2021-05-04       2021-05-04     2021-11-04 2021-11-04 2022-05-04
    2  2018-05-04 2021-11-04       2021-11-04     2022-05-04 2022-05-04 2022-11-04
    3  2018-05-04 2022-05-04       2022-05-04     2022-11-04 2022-11-04 2023-05-04
    4  2018-05-04 2022-11-04       2022-11-04     2023-05-04 2023-05-04 2023-11-04
    5  2018-05-04 2023-05-04       2023-05-04     2023-11-04 2023-11-04 2024-05-04
    6  2018-05-04 2023-11-04       2023-11-04     2024-05-04 2024-05-04 2024-11-04
    7  2018-05-04 2024-05-04       2024-05-04     2024-11-04 2024-11-04 2025-05-04
    8  2018-05-04 2024-11-04       2024-11-04     2025-05-04 2025-05-04 2025-11-04
    9  2018-05-04 2025-05-04       2025-05-04     2025-11-04 2025-11-04 2026-05-04


n_estimators = number of estimators: forest안의 트리 개수

In [9]:
# 06b-9. random forest validation 함수

def evaluate_rf_params(
    train_df,
    validation_df
):

    X_train = train_df[
        MODEL_FEATURES
    ]

    y_train = train_df[
        TARGET
    ]

    X_validation = validation_df[
        MODEL_FEATURES
    ]

    y_validation = validation_df[
        TARGET
    ]

    results = []


    for params in RF_CANDIDATES:

        model = RandomForestRegressor(
            n_estimators=300,
            max_depth=params[
                "max_depth"
            ],
            min_samples_leaf=params[
                "min_samples_leaf"
            ],
            max_features=params[
                "max_features"
            ],
            random_state=42,
            n_jobs=-1
        )

        model.fit(
            X_train,
            y_train
        )

        prediction = model.predict(
            X_validation
        )


        result_df = (
            validation_df[
                [
                    "signal_date",
                    "ticker",
                    TARGET
                ]
            ]
            .copy()
        )

        result_df[
            "prediction"
        ] = prediction


        rmse = np.sqrt(
            mean_squared_error(
                y_validation,
                prediction
            )
        )

        mae = mean_absolute_error(
            y_validation,
            prediction
        )

        r2 = r2_score(
            y_validation,
            prediction
        )


        ic_values = calculate_ic(
            result_df,
            "prediction",
            TARGET
        )


        results.append(
            {
                "max_depth": params[
                    "max_depth"
                ],
                "min_samples_leaf": params[
                    "min_samples_leaf"
                ],
                "max_features": params[
                    "max_features"
                ],
                "rmse": rmse,
                "mae": mae,
                "r2": r2,
                "mean_ic": (
                    ic_values.mean()
                    if len(ic_values) > 0
                    else np.nan
                ),
                "median_ic": (
                    np.median(
                        ic_values
                    )
                    if len(ic_values) > 0
                    else np.nan
                ),
                "ic_signals": len(
                    ic_values
                )
            }
        )


    return pd.DataFrame(
        results
    )

In [10]:
# 06b-10. fold 1 validation

fold = folds[0]


train_mask = (
    (
        supervised_dataset["signal_date"]
        < fold["train_end"]
    )
    &
    (
        supervised_dataset["next_execution_date"]
        <= fold["train_end"]
    )
)


validation_mask = (
    (
        supervised_dataset["signal_date"]
        >= fold["validation_start"]
    )
    &
    (
        supervised_dataset["signal_date"]
        < fold["validation_end"]
    )
    &
    (
        supervised_dataset["next_execution_date"]
        <= fold["validation_end"]
    )
)


train_df = (
    supervised_dataset.loc[
        train_mask
    ]
    .copy()
)

validation_df = (
    supervised_dataset.loc[
        validation_mask
    ]
    .copy()
)


print(
    "train:",
    train_df.shape
)

print(
    "validation:",
    validation_df.shape
)


rf_validation_results = (
    evaluate_rf_params(
        train_df,
        validation_df
    )
)


print(
    rf_validation_results
    .sort_values(
        [
            "rmse",
            "mean_ic"
        ],
        ascending=[
            True,
            False
        ]
    )
    .to_string(
        index=False
    )
)

train: (7791, 39)
validation: (1248, 39)
 max_depth  min_samples_leaf max_features     rmse      mae        r2   mean_ic  median_ic  ic_signals
       8.0            0.0100         sqrt 0.065331 0.043541  0.005866  0.060151   0.044322          25
       8.0            0.0010         sqrt 0.065441 0.043825  0.002511  0.089373   0.081489          25
       4.0            0.0010          0.5 0.065485 0.043946  0.001184  0.058623   0.121554          25
       4.0            0.0100         sqrt 0.065496 0.043826  0.000834 -0.010636   0.027185          25
      12.0            0.0100         sqrt 0.065498 0.043642  0.000767  0.057503   0.052581          25
      12.0            0.0025         sqrt 0.065583 0.043782 -0.001832  0.060776   0.061417          25
       8.0            0.0025         sqrt 0.065610 0.043876 -0.002654  0.073415   0.111068          25
       NaN            0.0100         sqrt 0.065650 0.043783 -0.003869  0.073196   0.068043          25
       8.0            0.0010    

In [11]:
# 06b-11. fold 1 best parameter 선택

best_rf_row = (
    rf_validation_results
    .sort_values(
        [
            "rmse",
            "mean_ic"
        ],
        ascending=[
            True,
            False
        ]
    )
    .iloc[0]
)


best_rf_params = {
    "max_depth": (
        None
        if pd.isna(
            best_rf_row[
                "max_depth"
            ]
        )
        else int(
            best_rf_row[
                "max_depth"
            ]
        )
    ),
    "min_samples_leaf": float(
        best_rf_row[
            "min_samples_leaf"
        ]
    ),
    "max_features": (
        best_rf_row[
            "max_features"
        ]
    )
}


print(
    "best params:",
    best_rf_params
)

print(
    best_rf_row
)

best params: {'max_depth': 8, 'min_samples_leaf': 0.01, 'max_features': 'sqrt'}
max_depth                8.0
min_samples_leaf        0.01
max_features            sqrt
rmse                0.065331
mae                 0.043541
r2                  0.005866
mean_ic             0.060151
median_ic           0.044322
ic_signals                25
Name: 18, dtype: object


In [12]:
# 06b-12. random forest 최종 후보 고정

from sklearn.model_selection import ParameterGrid


RF_PARAM_SPACE = {
    "max_depth": [
        4,
        8,
        12,
        None
    ],
    "min_samples_leaf": [
        0.005,
        0.01,
        0.015,
        0.02,
        0.03
    ],
    "max_features": [
        "sqrt",
        0.5,
        1.0
    ]
}


RF_CANDIDATES = list(
    ParameterGrid(
        RF_PARAM_SPACE
    )
)


print(
    "candidates:",
    len(RF_CANDIDATES)
)

candidates: 60


왜 0.03까지만?
현재 best 0.01에서:
0.015 → 약 117개 / leaf
0.020 → 약 156개 / leaf
0.030 → 약 234개 / leaf
까지, 즉 현재 최적점보다 최대 3배 강한 smoothing까지 확인

In [13]:
# 06b-13. fold 1 최종 validation

rf_validation_results = (
    evaluate_rf_params(
        train_df,
        validation_df
    )
)


print(
    rf_validation_results
    .sort_values(
        [
            "rmse",
            "mean_ic"
        ],
        ascending=[
            True,
            False
        ]
    )
    .head(20)
    .to_string(
        index=False
    )
)

 max_depth  min_samples_leaf max_features     rmse      mae       r2   mean_ic  median_ic  ic_signals
       4.0             0.015          1.0 0.065173 0.043463 0.010679  0.044651   0.017138          24
       4.0             0.020          1.0 0.065178 0.043482 0.010516  0.025175   0.039303          24
       4.0             0.015          0.5 0.065181 0.043563 0.010437  0.070833   0.065925          25
       4.0             0.020          0.5 0.065181 0.043605 0.010436  0.012748   0.010925          25
       8.0             0.015          1.0 0.065193 0.043547 0.010075  0.075453   0.120960          25
       4.0             0.030          1.0 0.065231 0.043636 0.008906  0.020229   0.049619          25
       8.0             0.015          0.5 0.065242 0.043587 0.008561  0.071858   0.072031          25
       4.0             0.030          0.5 0.065288 0.043806 0.007161 -0.015579  -0.035126          25
       8.0             0.020          0.5 0.065290 0.043728 0.007110  0.073155   0

In [14]:
# 06b-14. random forest parameter 확정

best_rf_row = (
    rf_validation_results
    .sort_values(
        [
            "rmse",
            "mean_ic"
        ],
        ascending=[
            True,
            False
        ]
    )
    .iloc[0]
)


best_rf_params = {
    "max_depth": (
        None
        if pd.isna(
            best_rf_row["max_depth"]
        )
        else int(
            best_rf_row["max_depth"]
        )
    ),
    "min_samples_leaf": float(
        best_rf_row[
            "min_samples_leaf"
        ]
    ),
    "max_features": best_rf_row[
        "max_features"
    ]
}


print(
    "best params:",
    best_rf_params
)

print(
    best_rf_row
)

best params: {'max_depth': 4, 'min_samples_leaf': 0.015, 'max_features': 1.0}
max_depth                4.0
min_samples_leaf       0.015
max_features             1.0
rmse                0.065173
mae                 0.043463
r2                  0.010679
mean_ic             0.044651
median_ic           0.017138
ic_signals                24
Name: 12, dtype: object


In [15]:
# 06b-15. parameter boundary 확인

leaf_upper_bound = (
    best_rf_params[
        "min_samples_leaf"
    ]
    == max(
        RF_PARAM_SPACE[
            "min_samples_leaf"
        ]
    )
)


print(
    "leaf upper bound:",
    leaf_upper_bound
)

leaf upper bound: False


In [16]:
# 06b-16. random forest 최종 search space

RF_PARAM_SPACE = {
    "max_depth": [
        1,
        2,
        3,
        4,
        6,
        8,
        12,
        None
    ],
    "min_samples_leaf": [
        0.005,
        0.01,
        0.015,
        0.02,
        0.03
    ],
    "max_features": [
        "sqrt",
        0.5,
        1.0
    ]
}

RF_CANDIDATES = list(
    ParameterGrid(
        RF_PARAM_SPACE
    )
)

print(
    "candidates:",
    len(RF_CANDIDATES)
)

candidates: 120


In [17]:
# 06b-17. fold 1 최종 validation

rf_validation_results = (
    evaluate_rf_params(
        train_df,
        validation_df
    )
)

print(
    rf_validation_results
    .sort_values(
        [
            "rmse",
            "mean_ic"
        ],
        ascending=[
            True,
            False
        ]
    )
    .head(20)
    .to_string(
        index=False
    )
)

 max_depth  min_samples_leaf max_features     rmse      mae       r2   mean_ic  median_ic  ic_signals
       3.0             0.015          1.0 0.065137 0.043621 0.011763  0.100788   0.101368          23
       3.0             0.020          1.0 0.065139 0.043616 0.011706  0.046379   0.048533          23
       4.0             0.015          1.0 0.065173 0.043463 0.010679  0.044651   0.017138          24
       4.0             0.020          1.0 0.065178 0.043482 0.010516  0.025175   0.039303          24
       4.0             0.015          0.5 0.065181 0.043563 0.010437  0.070833   0.065925          25
       4.0             0.020          0.5 0.065181 0.043605 0.010436  0.012748   0.010925          25
       8.0             0.015          1.0 0.065193 0.043547 0.010075  0.075453   0.120960          25
       3.0             0.020          0.5 0.065218 0.043734 0.009312  0.005036   0.002927          22
       6.0             0.015          1.0 0.065219 0.043487 0.009272  0.085289   0

In [18]:
# 06b-18. random forest 최종 parameter 선택

best_rf_row = (
    rf_validation_results
    .sort_values(
        [
            "rmse",
            "mean_ic"
        ],
        ascending=[
            True,
            False
        ]
    )
    .iloc[0]
)

best_rf_params = {
    "max_depth": (
        None
        if pd.isna(
            best_rf_row[
                "max_depth"
            ]
        )
        else int(
            best_rf_row[
                "max_depth"
            ]
        )
    ),
    "min_samples_leaf": float(
        best_rf_row[
            "min_samples_leaf"
        ]
    ),
    "max_features": best_rf_row[
        "max_features"
    ]
}

print(
    "best params:",
    best_rf_params
)

print(
    best_rf_row
)

best params: {'max_depth': 3, 'min_samples_leaf': 0.015, 'max_features': 1.0}
max_depth                3.0
min_samples_leaf       0.015
max_features             1.0
rmse                0.065137
mae                 0.043621
r2                  0.011763
mean_ic             0.100788
median_ic           0.101368
ic_signals                23
Name: 42, dtype: object


In [19]:
# 06b-19. parameter boundary 진단

depth_minimum = (
    best_rf_params[
        "max_depth"
    ]
    == 1
)

leaf_upper_bound = (
    best_rf_params[
        "min_samples_leaf"
    ]
    == max(
        RF_PARAM_SPACE[
            "min_samples_leaf"
        ]
    )
)

leaf_lower_bound = (
    best_rf_params[
        "min_samples_leaf"
    ]
    == min(
        RF_PARAM_SPACE[
            "min_samples_leaf"
        ]
    )
)

print(
    "depth minimum:",
    depth_minimum
)

print(
    "leaf lower bound:",
    leaf_lower_bound
)

print(
    "leaf upper bound:",
    leaf_upper_bound
)

depth minimum: False
leaf lower bound: False
leaf upper bound: False


각 fold마다 동일한 120개 search space에서 validation으로 best parameter를 새로 선택하고 → train+validation으로 재학습 → 해당 fold test 평가

In [ ]:
# 06b-20. random forest walk-forward 실행

rf_fold_rows = []
rf_oos_frames = []


for fold in folds:

    train_mask = (
        (
            supervised_dataset["signal_date"]
            < fold["train_end"]
        )
        &
        (
            supervised_dataset["next_execution_date"]
            <= fold["train_end"]
        )
    )

    validation_mask = (
        (
            supervised_dataset["signal_date"]
            >= fold["validation_start"]
        )
        &
        (
            supervised_dataset["signal_date"]
            < fold["validation_end"]
        )
        &
        (
            supervised_dataset["next_execution_date"]
            <= fold["validation_end"]
        )
    )

    test_mask = (
        (
            supervised_dataset["signal_date"]
            >= fold["test_start"]
        )
        &
        (
            supervised_dataset["signal_date"]
            < fold["test_end"]
        )
        &
        (
            supervised_dataset["next_execution_date"]
            <= fold["test_end"]
        )
    )


    train_df = (
        supervised_dataset.loc[
            train_mask
        ]
        .copy()
    )

    validation_df = (
        supervised_dataset.loc[
            validation_mask
        ]
        .copy()
    )

    test_df = (
        supervised_dataset.loc[
            test_mask
        ]
        .copy()
    )


    validation_results = (
        evaluate_rf_params(
            train_df,
            validation_df
        )
    )

    best_row = (
        validation_results
        .sort_values(
            "rmse"
        )
        .iloc[0]
    )


    best_max_depth = (
        None
        if pd.isna(
            best_row["max_depth"]
        )
        else int(
            best_row["max_depth"]
        )
    )

    best_min_samples_leaf = float(
        best_row[
            "min_samples_leaf"
        ]
    )

    best_max_features = (
        best_row[
            "max_features"
        ]
    )


    train_validation_mask = (
        (
            supervised_dataset["signal_date"]
            < fold["test_start"]
        )
        &
        (
            supervised_dataset["next_execution_date"]
            <= fold["test_start"]
        )
    )

    train_validation_df = (
        supervised_dataset.loc[
            train_validation_mask
        ]
        .copy()
    )


    X_train_validation = (
        train_validation_df[
            MODEL_FEATURES
        ]
    )

    y_train_validation = (
        train_validation_df[
            TARGET
        ]
    )

    X_test = (
        test_df[
            MODEL_FEATURES
        ]
    )

    y_test = (
        test_df[
            TARGET
        ]
    )


    rf_model = RandomForestRegressor(
        n_estimators=300,
        max_depth=best_max_depth,
        min_samples_leaf=(
            best_min_samples_leaf
        ),
        max_features=best_max_features,
        random_state=42,
        n_jobs=-1
    )

    rf_model.fit(
        X_train_validation,
        y_train_validation
    )


    rf_pred = rf_model.predict(
        X_test
    )


    dummy_model = DummyRegressor(
        strategy="mean"
    )

    dummy_model.fit(
        X_train_validation,
        y_train_validation
    )

    dummy_pred = dummy_model.predict(
        X_test
    )


    test_result = (
        test_df[
            [
                "signal_date",
                "execution_date",
                "ticker",
                "name",
                TARGET
            ]
        ]
        .copy()
    )

    test_result[
        "prediction"
    ] = rf_pred

    test_result[
        "dummy_prediction"
    ] = dummy_pred

    test_result[
        "fold"
    ] = fold["fold"]


    ic_values = calculate_ic(
        test_result,
        "prediction",
        TARGET
    )


    rf_rmse = np.sqrt(
        mean_squared_error(
            y_test,
            rf_pred
        )
    )

    rf_mae = mean_absolute_error(
        y_test,
        rf_pred
    )

    rf_r2 = r2_score(
        y_test,
        rf_pred
    )


    dummy_rmse = np.sqrt(
        mean_squared_error(
            y_test,
            dummy_pred
        )
    )

    dummy_mae = mean_absolute_error(
        y_test,
        dummy_pred
    )

    dummy_r2 = r2_score(
        y_test,
        dummy_pred
    )


    rf_fold_rows.append(
        {
            "fold": fold["fold"],
            "max_depth": best_max_depth,
            "min_samples_leaf":
                best_min_samples_leaf,
            "max_features":
                best_max_features,
            "validation_rmse":
                best_row["rmse"],
            "test_start":
                fold["test_start"],
            "test_end":
                fold["test_end"],
            "test_rows":
                len(test_df),
            "test_signals":
                test_df[
                    "signal_date"
                ].nunique(),
            "rf_rmse":
                rf_rmse,
            "rf_mae":
                rf_mae,
            "rf_r2":
                rf_r2,
            "mean_ic": (
                ic_values.mean()
                if len(ic_values) > 0
                else np.nan
            ),
            "median_ic": (
                np.median(
                    ic_values
                )
                if len(ic_values) > 0
                else np.nan
            ),
            "ic_signals":
                len(ic_values),
            "dummy_rmse":
                dummy_rmse,
            "dummy_mae":
                dummy_mae,
            "dummy_r2":
                dummy_r2
        }
    )


    rf_oos_frames.append(
        test_result
    )


    print(
        "fold",
        fold["fold"],
        "done"
    )

fold 1 done
fold 2 done
fold 3 done
fold 4 done
fold 5 done
fold 6 done
fold 7 done


각 fold의 best parameter가 얼마나 바뀌는가
RF가 Dummy보다 몇 fold에서 나은가
IC 부호가 기간마다 얼마나 안정적인가

In [ ]:
# 06b-21. random forest fold 결과

rf_fold_results = pd.DataFrame(
    rf_fold_rows
)

rf_fold_results[
    "rmse_improvement"
] = (
    rf_fold_results[
        "dummy_rmse"
    ]
    - rf_fold_results[
        "rf_rmse"
    ]
)

rf_fold_results[
    "rf_better"
] = (
    rf_fold_results[
        "rf_rmse"
    ]
    <
    rf_fold_results[
        "dummy_rmse"
    ]
)


print(
    rf_fold_results.to_string(
        index=False
    )
)

In [ ]:
# 06b-22. random forest oos prediction 결합

rf_oos_predictions = (
    pd.concat(
        rf_oos_frames,
        ignore_index=True
    )
    .sort_values(
        [
            "signal_date",
            "ticker"
        ]
    )
    .reset_index(
        drop=True
    )
)


print(
    "shape:",
    rf_oos_predictions.shape
)

print(
    "signals:",
    rf_oos_predictions[
        "signal_date"
    ].nunique()
)

print(
    "start:",
    rf_oos_predictions[
        "signal_date"
    ].min()
)

print(
    "end:",
    rf_oos_predictions[
        "signal_date"
    ].max()
)

print(
    "duplicates:",
    rf_oos_predictions[
        [
            "signal_date",
            "ticker"
        ]
    ]
    .duplicated()
    .sum()
)

In [ ]:
# 06b-23. random forest 전체 oos 성능

pooled_rf_rmse = np.sqrt(
    mean_squared_error(
        rf_oos_predictions[
            TARGET
        ],
        rf_oos_predictions[
            "prediction"
        ]
    )
)

pooled_rf_mae = mean_absolute_error(
    rf_oos_predictions[
        TARGET
    ],
    rf_oos_predictions[
        "prediction"
    ]
)

pooled_rf_r2 = r2_score(
    rf_oos_predictions[
        TARGET
    ],
    rf_oos_predictions[
        "prediction"
    ]
)

pooled_rf_ic = calculate_ic(
    rf_oos_predictions,
    "prediction",
    TARGET
)


print(
    "pooled rmse:",
    pooled_rf_rmse
)

print(
    "pooled mae:",
    pooled_rf_mae
)

print(
    "pooled r2:",
    pooled_rf_r2
)

print(
    "mean ic:",
    pooled_rf_ic.mean()
)

print(
    "median ic:",
    np.median(
        pooled_rf_ic
    )
)

print(
    "ic signals:",
    len(
        pooled_rf_ic
    )
)

print(
    "rf better folds:",
    rf_fold_results[
        "rf_better"
    ].sum(),
    "/",
    len(
        rf_fold_results
    )
)

In [ ]:
# 06b-24. random forest prediction 분산

rf_prediction_stats = (
    rf_oos_predictions
    .groupby(
        "signal_date"
    )
    .agg(
        prediction_std=(
            "prediction",
            "std"
        ),
        target_std=(
            TARGET,
            "std"
        )
    )
)


print(
    rf_prediction_stats.describe()
)